# Canonical Model 06: Ensemble Uncertainty & IES Diagnostics

This notebook works through the **per-cycle ensemble diagnostics** to look at every time: prior Monte Carlo and prior-data conflict, phi distribution & contribution, parameters at bounds, posterior forecast uncertainty, and **spatial K-field ("property pattern") maps**.

It calibrates a *spatially-varying* K field (one geostatistical parameter per Voronoi cell), so the maps in section 8 answer the workshop's question: *property patterns -- plausible or laughable?* Every plot offers both a `plotly` and a `matplotlib` backend.

In [1]:
import sys
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import pandas as pd
import myflopy as mf
from canonical_notebook_style import notebook_header
from myflopy.modflow.mf6.canonical_calibration import build_canonical_calibration_demo
from myflopy.modflow.mf6.grid.plotting import build_choropleth

notebook_header('06', 'Ensemble Uncertainty & IES Diagnostics',
                'Prior MC, prior-data conflict, phi & weight diagnostics, forecast uncertainty, K-field maps.')

## 1. Build and declare the PEST problem (pilot-point K)

In [ ]:
import flopy

flopy.utils.gridintersect

In [2]:
import shutil
artifact_root = Path('../artifacts/canonical_pest_ppk')
shutil.rmtree(artifact_root, ignore_errors=True)  # clean rebuild: a stale pest/ template inside the model dir makes PstFrom recurse
artifact_root.mkdir(parents=True, exist_ok=True)

# A SMOOTH synthetic truth K field (a homogeneous base x a smooth anomaly) on
# the two unconfined layers, with a flat-base START -- IES recovers the spatial
# pattern AND the prior brackets the data. (The model's own sharp paleochannel K
# is a near-extreme low-head configuration a smooth prior cannot bracket, which
# is why we use a smooth synthetic anomaly here.)
demo = build_canonical_calibration_demo(artifact_root / 'model', synthetic_k=True,
                                        synthetic_k_base=20.0, start_k_layers=(0, 1))

# No workspace=: defaults to <model workspace>/pest/canonical_ppk (see
# demo.model.pest_runs in the last cell).
cal = demo.model.pest('canonical_ppk', start_datetime='2024-01-01')
# Pilot-point K: a sparse net of points (~8 cells apart) on each of the two
# unconfined layers; their multipliers are interpolated to every cell by inverse-
# distance weighting at forward-run time. Parsimonious (a handful of parameters
# per layer) and the supported sparse-field style on Voronoi grids -- it recovers
# a cleaner field from sparse heads than a 2500-cell grid. `capture=True` records
# every realization's resolved K field so we can map it.
cal.parameterize('k', style='pilotpoints', pp_space=8, layers=[0, 1],
                 bounds=(0.05, 20.0), physical=(0.001, 300.0), capture=True)
cal.parameterize('recharge', style='constant', bounds=(0.3, 3.0), physical=(0.0, 1e-2))
cal.observe(demo.head_targets)
cal.forecast(demo.forecast_targets)
pst = cal.build('canonical_ppk.pst', noptmax=0)
print('adjustable parameters:', pst.npar_adj, '| nonzero obs:', pst.nnz_obs)

VoronoiGrid initializing.


Voronoi grid initialized.


getting connectivity properties (iac, ja, cl12, hwva, nja)


C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\flopy\mf6\mfmodel.py:278: DeprecationWarning: This method is for internal use only and will be deprecated.
  warnings.warn(


writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_gwf...
  writing model viz_prt_master...
    writing model name file...
    writing package disv...


...

    Solving:  Stress period:     6    Time step:     1


    Solving:  Stress period:     6    Time step:     2


 
 Run end date and time (yyyy/mm/dd hh:mm:ss): 2026/06/24 11:32:53
 Elapsed run time:  6.767 Seconds
 
 Normal termination of simulation.

Success is:  True


error trying to read input file with tpl file:input file EOF, tpl file line 66, in file line 65
64 pars added from template file .\kl0_pp.csv.tpl
error trying to read input file with tpl file:input file EOF, tpl file line 66, in file line 65
64 pars added from template file .\kl1_pp.csv.tpl
adjustable parameters: 129 | nonzero obs: 96


C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\pyemu\utils\pst_from.py:1281: PyemuWarning: add_py_function(): _write_head_target_csv already in forward run python functions, not overriding here, original will be maintained


## 2. Prior Monte Carlo -- "early and often"

Evaluate the **prior** parameter ensemble once (`cal.prior(...)`, NOPTMAX = -1) before history matching: is the prior wide enough to explain the data, and is any observation *outside* what the prior can produce? Grey "spaghetti" is the prior simulated ensemble; red triangles are the measured values. **What to look for:** measured points inside the grey band.

In [3]:
# RUN_IES gates the heavy ensemble work in this notebook (prior MC + IES).
# Saved outputs below are from a completed run. Set RUN_IES = True to
# regenerate them (prior MC ~1-2 min, IES ~6-8 min with 12 workers).
RUN_IES = False
if RUN_IES:
    pm = cal.prior(reals=40, workers=12)
    display(pm.plot_prior_vs_obs())
    display(pm.plot_prior_vs_obs(backend='matplotlib'))
else:
    pm = None


2026-06-24 11:33:08,590 - MainProcess - INFO - Reserved port 4375 for process 8804


<Figure size 800x3120 with 12 Axes>

## 3. Prior-data conflict

Where a measured value lies *outside* the prior predictive band, prior and data disagree before calibration starts -- usually a too-narrow prior, a wrong weight, or missing physics. **What to look for:** groups with a high % in conflict; investigate before trusting history matching.

In [4]:
if pm is not None:
    conflict = pm.conflict()
    print('observations in conflict:', int(conflict['in_conflict'].sum()), 'of', len(conflict))
    display(pm.plot_conflict())
    display(conflict[conflict['in_conflict']].head(10))


observations in conflict: 32 of 96


,obgnme,time,measured,prior_lo,prior_hi,prior_mean,in_conflict
oname:hds_otype:lst_usecol:obs_01_per:5,oname:hds_otype:lst_usecol:obs_01,5.0,104.678896,93.947685,104.671852,99.222760,True
oname:hds_otype:lst_usecol:obs_04_per:0,oname:hds_otype:lst_usecol:obs_04,0.0,107.647944,96.293447,106.161544,101.538486,True
oname:hds_otype:lst_usecol:obs_04_per:1,oname:hds_otype:lst_usecol:obs_04,1.0,108.272663,96.848984,106.790305,102.157055,True
oname:hds_otype:lst_usecol:obs_04_per:2,oname:hds_otype:lst_usecol:obs_04,2.0,108.460326,97.118243,107.011087,102.448719,True
oname:hds_otype:lst_usecol:obs_04_per:3,oname:hds_otype:lst_usecol:obs_04,3.0,108.484617,97.177116,106.911933,102.426920,True
oname:hds_otype:lst_usecol:obs_04_per:4,oname:hds_otype:lst_usecol:obs_04,4.0,108.426294,97.130299,106.743522,102.294318,True
oname:hds_otype:lst_usecol:obs_04_per:5,oname:hds_otype:lst_usecol:obs_04,5.0,108.328387,97.002722,106.554496,102.122056,True
oname:hds_otype:lst_usecol:obs_07_per:0,oname:hds_otype:lst_usecol:obs_07,0.0,106.674071,94.775594,104.698793,99.418244,True
oname:hds_otype:lst_usecol:obs_07_per:1,oname:hds_otype:lst_usecol:obs_07,1.0,107.228724,95.334969,105.347615,100.013793,True
oname:hds_otype:lst_usecol:obs_07_per:2,oname:hds_otype:lst_usecol:obs_07,2.0,107.561646,95.649681,105.693113,100.369020,True


## 4. Run PESTPP-IES

A grid-K ensemble (thousands of parameters) with parallel agents -- this is a *go-get-coffee* cell (~minutes). Set `RUN_IES = False` to skip.

In [5]:
if RUN_IES:
    ies = cal.run_ies(reals=30, iterations=3, workers=12)
    print(ies.settings)
else:
    ies = None
    print('Skipped: set RUN_IES = True (top of the prior-MC cell) to run PESTPP-IES.')


2026-06-24 11:34:24,555 - MainProcess - INFO - Reserved port 4818 for process 8804


PESTPP-IES run: canonical_ppk
  workspace   : ..\artifacts\canonical_pest_ppk\model\viz_prt_master\pest\canonical_ppk_ies_master
  realizations: 30
  noptmax     : 3   iterations on disk: [0, 1, 2, 3]
  observations: 96 nonzero-weight   forecasts: 6
  noise ensemble: yes
  ies options : {'ies_num_reals': 30}


## 5. Phi convergence and distribution (the "pepsi challenge")

`plot_phi` shows the objective function dropping across iterations; `plot_phi_distribution` overlays prior vs posterior phi *histograms* (log scale). **What to look for:** the posterior histogram shifted left of the prior, and a posterior phi near the observation count (a phi far below that is over-fitting).

In [6]:
if ies is not None:
    display(ies.plot_phi())
    display(ies.plot_phi_distribution())
    display(ies.plot_phi_distribution(backend='matplotlib'))

<Figure size 650x400 with 1 Axes>

## 6. Phi contribution by group, and parameters at bounds

`plot_phi_contributions` breaks total phi down by observation group (is one group dominating the misfit?); `parameters_at_bounds` reports the fraction of each parameter group pinned at a bound (are bounds too tight?). **What to look for:** a balanced phi across groups, and few parameters stuck at bounds.

In [7]:
if ies is not None:
    display(ies.plot_phi_contributions())
    display(ies.parameters_at_bounds())
    display(ies.plot_parameters_at_bounds())

,n_parameters,n_at_lower,n_at_upper,pct_at_bound
pargp,,,,
kl0,64,0,0,0.0
kl1,64,0,0,0.0
recharge,1,0,0,0.0


## 7. Forecast uncertainty -- "uncertainty analysis for free"

The posterior distribution of the prediction of interest (a down-valley head near the lake). **What to look for:** history matching should *narrow* the forecast relative to the prior -- but a good fit is not a good prediction, so value the posterior spread.

In [8]:
if ies is not None:
    display(ies.forecasts())
    name = ies.forecast_names[0]
    display(ies.forecast(name).plot())

,prior_mean,prior_std,prior_p05,prior_p50,prior_p95,posterior_mean,posterior_std,posterior_p05,posterior_p50,posterior_p95,truth,uncertainty_reduction
forecast,,,,,,,,,,,,
oname:fore1_otype:lst_usecol:fore_lakehead_per:0,103.454027,2.663380,98.647572,103.002416,107.176523,109.384285,0.301622,109.045053,109.316179,110.009310,108.526230,0.886752
oname:fore1_otype:lst_usecol:fore_lakehead_per:1,104.052252,2.636797,99.273016,103.643849,107.772081,109.938092,0.290727,109.616480,109.874746,110.531356,109.059245,0.889742
oname:fore1_otype:lst_usecol:fore_lakehead_per:2,104.309872,2.584449,99.597471,103.957892,107.961726,110.062765,0.277008,109.764321,110.004262,110.620208,109.155344,0.892818
oname:fore1_otype:lst_usecol:fore_lakehead_per:3,104.291391,2.527437,99.701381,103.956860,107.864244,110.027162,0.270089,109.737575,109.972983,110.576166,109.129935,0.893137
oname:fore1_otype:lst_usecol:fore_lakehead_per:4,104.199579,2.510880,99.664027,103.841379,107.716408,109.967015,0.277392,109.676092,109.903156,110.545175,109.110957,0.889524
oname:fore1_otype:lst_usecol:fore_lakehead_per:5,104.052077,2.513373,99.537291,103.679176,107.544627,109.855777,0.282915,109.561697,109.796427,110.457192,109.056083,0.887436


## 8. Property patterns -- did IES recover the K field?

*"Property patterns: plausible or laughable?"* Because we declared `capture=True`, every realization's resolved K field is recorded, so we can map it. Compare the known **truth** K against the **posterior mean** (what IES recovered) and the **posterior std** (where K is still uncertain).

**What to look for:** with only head observations, fine K structure *cannot* be fully resolved -- the posterior mean is a smooth, plausible field that adjusts K **near the observation wells** and stays close to the prior elsewhere. The std map shows exactly where the data does (and does not) constrain K. A posterior that reproduced the sharp truth everywhere from sparse heads would be *too good to be true*.

In [9]:
if ies is not None:
    layer = 0
    truth_k = list(demo.truth_k[layer])                       # known truth (layer 1)
    display(build_choropleth(demo.model.vor, custom_zs=truth_k, layer=layer))
    display(ies.plot_field('k', stat='mean', which='posterior', layer=layer))   # recovered
    display(ies.plot_field('k', stat='std', layer=layer))                       # uncertainty
    display(ies.plot_field('k', stat='change', layer=layer, backend='matplotlib'))  # prior->post
    display(ies.field('k', layer=layer).head())

<Figure size 700x600 with 2 Axes>

,cell,prior_mean,prior_std,posterior_mean,posterior_std,base,change
0,0,51.517802,28.775427,78.364497,22.985030,57.907175,1.521115
1,1,50.924270,36.014816,93.572598,30.973302,69.301970,1.837485
2,2,50.769752,46.541843,114.221544,41.357839,84.661012,2.249795
3,3,50.466161,51.785254,123.149646,46.138500,91.370527,2.440242
4,4,50.354195,54.436096,127.535022,48.518767,94.664167,2.532759


## 9. The single best realization, and the report bundle

Carry the **base** (minimum-error-variance) realization forward, never the lowest-phi one. `report(...)` bundles the diagnostics into one HTML file.

In [10]:
if ies is not None:
    print('recommended realization:', ies.best())
    report_path = ies.report(artifact_root / 'uncertainty_review.html')
    print('wrote', report_path)

recommended realization: base


wrote ..\artifacts\canonical_pest_ppk\uncertainty_review.html


## Interpretation checklist (run this every cycle)

- **Prior Monte Carlo first:** is the prior wide enough, and is any observation in conflict with it?
- **Phi:** did it drop, and is the posterior phi near (not far below) the observation count?
- **Phi by group / parameters at bounds:** is one group dominating, or are bounds too tight?
- **Forecast:** value the posterior *spread* -- a good fit is not a good prediction.
- **Property patterns:** are the posterior K maps geologically plausible? Trust them only where the data constrains them (low posterior std).
- Carry the **base** realization forward.

## Your PEST runs live with the model

Because the calibration workspace defaults to `<model workspace>/pest/<name>`,
every PEST run done on this model is discoverable straight from the model via
`model.pest_runs` (or `run.pest_runs` for a loaded run) -- no need to remember
where the master directories were written. Reopen any of them for the full
`IesResults` review **without re-running**.

In [11]:
# Discover every PEST run done on this model, and reopen one for review.
runs = demo.model.pest_runs
for run in runs:
    print(run)

this_run = next((r for r in runs if r.name == cal.name and r.kinds), None)
if this_run is not None:
    review = this_run.review()          # -> IesResults (identical to a fresh run_ies)
    display(review.plot_phi())
else:
    print('No completed PEST runs yet -- set RUN_IES = True above and re-run.')

<PestRun 'canonical_ppk' on viz_prt_master: ies, prior>
